# 💎 Diamond Price Analysis & Prediction
### A complete Data Science mini-project: cleaning → EDA → statistics → machine learning

**Goal:** Understand what factors drive diamond prices, and build a model that predicts price from a diamond's characteristics.

**Skills demonstrated:** pandas, data cleaning, exploratory data analysis, data visualization, hypothesis testing, linear regression, random forest, model evaluation.

**Dataset:** ~54,000 real diamonds (built into the seaborn library) with features like carat, cut, color, clarity, and price.

> Run each cell in order (Shift+Enter). Read the comments — they explain *why*, not just *what*.


In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)


## 2. Load the data

In [ ]:
df = sns.load_dataset('diamonds')
print("Shape:", df.shape)
df.head()


## 3. Simulate real-world messiness (then clean it)

Real datasets are rarely this clean. To practice actual data-cleaning skills (which is most of a Data Analyst's job),
we'll intentionally introduce missing values, invalid entries, and duplicates — then clean them properly.


In [ ]:
np.random.seed(42)
dirty_df = df.copy()
dirty_idx = np.random.choice(dirty_df.index, size=50, replace=False)

dirty_df.loc[dirty_idx[:25], 'carat'] = np.nan      # missing values
dirty_df.loc[dirty_idx[25:], 'price'] = -1          # invalid negative prices
dirty_df = pd.concat([dirty_df, dirty_df.iloc[:20]]) # duplicate rows

print("Before cleaning:", dirty_df.shape)
print("Missing values:\n", dirty_df.isnull().sum()[dirty_df.isnull().sum() > 0])
print("Invalid prices:", (dirty_df['price'] < 0).sum())
print("Duplicate rows:", dirty_df.duplicated().sum())


In [ ]:
# Clean it step by step
df_clean = dirty_df.drop_duplicates()                              # remove duplicates
df_clean = df_clean[df_clean['price'] > 0]                         # drop invalid prices
df_clean['carat'] = df_clean['carat'].fillna(df_clean['carat'].median())  # fill missing with median

df_clean = df_clean.reset_index(drop=True)

print("After cleaning:", df_clean.shape)
print("Remaining missing values:", df_clean.isnull().sum().sum())

df = df_clean  # use the cleaned version from here on


## 4. Exploratory Data Analysis (EDA)

Let's understand the shape of the data before modeling anything.


In [ ]:
df['price'].describe()


In [ ]:
plt.figure()
sns.histplot(df['price'], bins=50, kde=True)
plt.title('Distribution of Diamond Prices')
plt.xlabel('Price ($)')
plt.show()


**Observation:** Price is heavily right-skewed — most diamonds are cheap, with a long tail of expensive ones. This is common for price/income data.

In [ ]:
plt.figure()
sns.boxplot(data=df, x='cut', y='price', order=['Fair','Good','Very Good','Premium','Ideal'])
plt.title('Price by Cut Quality')
plt.show()


**Interesting finding:** 'Ideal' cut diamonds actually have a *lower* average price than 'Premium' or 'Fair' cuts.
This seems counterintuitive — better cut should mean higher price, right? We'll investigate this with correlation analysis next
(hint: it's a classic confounding-variable story involving carat size).

In [ ]:
corr = df[['carat','depth','table','price','x','y','z']].corr()

plt.figure(figsize=(7,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

corr['price'].sort_values(ascending=False)


**Explanation of the earlier puzzle:** `carat` (size) correlates most strongly with price (~0.92), far more than cut quality does.
Buyers of large diamonds often prioritize size over cut precision, which is why average price-by-cut doesn't behave as expected —
carat size is a confounding variable. This is a great example of why correlation-checking matters before drawing conclusions from group averages.

## 5. Hypothesis Testing

**Question:** Is the price difference between 'Ideal' and 'Fair' cut diamonds statistically significant, or could it be due to random chance?

We'll use an independent two-sample t-test.


In [ ]:
ideal_prices = df[df['cut'] == 'Ideal']['price']
fair_prices = df[df['cut'] == 'Fair']['price']

t_stat, p_value = stats.ttest_ind(ideal_prices, fair_prices, equal_var=False)
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.2e}")

if p_value < 0.05:
    print("=> Statistically significant difference (p < 0.05)")
else:
    print("=> No statistically significant difference")


**Interpretation:** With p < 0.05, the price difference is statistically significant — it's not random noise. But as shown above, this doesn't mean cut *causes* the price difference; carat size is the bigger driver.

## 6. Predictive Modeling

Now let's build models to predict `price` from the diamond's physical characteristics.
We'll compare a simple **Linear Regression** against a **Random Forest** (which can capture non-linear relationships).


In [ ]:
features_numeric = ['carat', 'depth', 'table', 'x', 'y', 'z']
features_categorical = ['cut', 'color', 'clarity']

X = pd.get_dummies(df[features_numeric + features_categorical], columns=features_categorical, drop_first=True)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape, " Test size:", X_test.shape)


In [ ]:
# Linear Regression (baseline model)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

print("Linear Regression:")
print("  R² score:", round(r2_score(y_test, lr_preds), 4))
print("  MAE ($):", round(mean_absolute_error(y_test, lr_preds), 2))


In [ ]:
# Random Forest (more powerful, non-linear model)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print("Random Forest:")
print("  R² score:", round(r2_score(y_test, rf_preds), 4))
print("  MAE ($):", round(mean_absolute_error(y_test, rf_preds), 2))


In [ ]:
# Which features matter most to the Random Forest model?
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure()
importances.plot(kind='barh')
plt.title('Top 10 Most Important Features (Random Forest)')
plt.gca().invert_yaxis()
plt.show()


## 7. Conclusion / Summary

Write your own 3-5 sentence summary here as if reporting to a manager. For example:

- Diamond prices are primarily driven by **carat weight**, followed by physical dimensions (x, y, z).
- Contrary to intuition, cut quality alone is *not* the strongest price driver — carat size dominates.
- The price difference between 'Ideal' and 'Fair' cuts is statistically significant but small relative to the effect of carat size.
- A Random Forest model explains about **98% of the variance** in diamond prices (R² ≈ 0.98), far outperforming plain linear regression, meaning the price relationship is non-linear.
- **Next step ideas:** try gradient boosting (XGBoost/LightGBM), tune hyperparameters, or deploy this model behind a simple price-estimator web app.

---
### 💡 How to extend this project further (for your resume):
1. Add a **Streamlit** app where a user inputs diamond specs and gets a predicted price.
2. Try feature engineering: create a `volume = x*y*z` feature and see if it improves the model.
3. Push this notebook to GitHub with a clear `README.md` explaining the problem, approach, and findings.
4. Write a short LinkedIn post about your top 2 findings — recruiters *do* look at this.
